In [161]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import settings
from src.aggregations import territorial, person, labor,cadunico
import pandas as pd 
import numpy as np
from pathlib import Path
import importlib

importlib.reload(settings)
importlib.reload(territorial)
importlib.reload(person)
importlib.reload(labor)
importlib.reload(cadunico)

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", None)

In [ ]:
df_cubo_old = pd.read_excel(settings.CUBO_PATH)

In [2]:
df_cubo = pd.read_excel(settings.DATA_PATH /'input_data' /'dados_cubo_final_v0__2026-05-28_16-48.xlsx')

In [ ]:
# Section 1
# bignumber1.csv

# Valor executado por estado x municipio abs / percentual 
df_cubo_est = df_cubo[df_cubo['tipo_ente'] == 'ESTADO']
df_cubo_mun = df_cubo[df_cubo['tipo_ente'] == 'MUNICIPIO']

# Totais executados com ceil
valor_estados = np.ceil(df_cubo_est["valor_transacao"].sum())
valor_municipios = np.ceil(df_cubo_mun["valor_transacao"].sum())

# Valores de referência
total_estados = 1_510_000_000
total_municipios = 1_490_000_000

df_execucao = pd.DataFrame({
    "Estados_DF": [valor_estados],
    "Municipios_DF": [valor_municipios],
    "perc_executado_estados": [valor_estados / total_estados],
    "perc_executado_municipios": [valor_municipios / total_municipios],
})

df_execucao

,Estados_DF,Municipios_DF,perc_executado_estados,perc_executado_municipios
0,1.450514e+09,1.395481e+09,0.960606,0.936565


In [29]:
df_aux = pd.read_parquet(settings.DATA_PATH / 'input_data'/ 'non-public' / 'df_aux_cubo.parquet')

In [42]:
df_execucao.to_csv('s1_bn1.csv')

In [3]:
# executed_value_by.csv
df_states       = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='ESTADO')
df_municipality = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_uf           = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='UF')

# df_states.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_state.csv')
# df_municipality.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_municipality.csv')
# df_uf.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_uf.csv')

In [53]:
df_uf_cut = df_uf[['uf','valor_executado_rs','valor_executado_perc', 'min_valor', 'mediana_valor', 'max_valor', 'media_valor','valor_executado_percapita', 'qtde_contemplados']]

In [173]:
df_uf_cut = territorial.criar_df_uf_cut_from_aux(df_aux=df_aux)
df_uf_cut.to_csv(settings.DATA_PATH_SECTION2 / 'resumo_valores_uf_estado.csv')

In [172]:
df_uf_cut

,uf,macrorregiao,valor_executado_rs,valor_executado_perc,valor_executado_perc_regiao,min_valor,mediana_valor,max_valor,media_valor,media_aparada_1pct_valor,valor_executado_percapita,qtde_contemplados,perc_qtde_contemplados_regiao
0,GO,Centro-Oeste,"105,504,440.400",0.037,0.444,400.000,"5,000.000","2,000,000.000","24,422.324","20,189.126",14.353,4320,0.463
1,MT,Centro-Oeste,"48,393,906.410",0.017,0.204,386.700,"5,000.000","1,800,000.000","19,882.459","16,145.757",12.614,2434,0.261
2,MS,Centro-Oeste,"42,351,939.740",0.015,0.178,378.000,"6,308.820","2,000,000.000","19,356.462","14,504.693",14.595,2188,0.234
3,DF,Centro-Oeste,"41,170,700.290",0.014,0.173,500.000,"26,569.700","2,000,000.000","104,760.052","85,294.542",13.803,393,0.042
4,BA,Nordeste,"176,909,114.840",0.062,0.208,400.000,"3,100.000","2,886,960.000","10,109.092","7,772.333",11.913,17500,0.220
5,PE,Nordeste,"141,508,869.190",0.050,0.166,400.000,"3,000.000","2,356,525.570","10,503.924","8,896.522",14.835,13472,0.170
6,CE,Nordeste,"136,622,936.070",0.048,0.160,400.000,"4,000.000","3,228,537.250","16,548.321","12,580.850",14.796,8256,0.104
7,MA,Nordeste,"114,921,030.050",0.040,0.135,400.000,"3,000.000","7,792,386.600","13,736.676","9,387.573",16.392,8366,0.105
8,PB,Nordeste,"70,229,086.670",0.025,0.082,380.000,"2,000.000","1,300,000.000","6,651.741","5,440.519",16.943,10558,0.133
9,AL,Nordeste,"60,294,994.130",0.021,0.071,390.000,"3,000.000","2,634,948.390","9,453.590","7,441.977",18.725,6378,0.080


In [166]:
df_uf_cut[['uf', 'media_aparada_1pct_valor']].sort_values(by='media_aparada_1pct_valor', ascending=False)

,uf,media_aparada_1pct_valor
3,DF,"85,294.542"
14,AM,"29,965.943"
20,SP,"26,023.174"
22,RJ,"25,915.248"
17,AP,"25,473.072"
23,ES,"23,651.241"
18,RR,"21,714.912"
24,PR,"20,920.838"
0,GO,"20,189.126"
25,RS,"18,276.764"


In [71]:
df_uf_cut.to_excel('tabela_uf.xlsx')
df_uf_cut.to_csv('tabela_uf.csv')
df_uf_cut.to_parquet('tabela_uf.parquet')


In [132]:
# aggregate_faixa_valor_ju_wide_by_uf.csv
df_uf_new = territorial.aggregate_faixa_valor_ju_wide_by_uf(df_cubo=df_cubo)
# df_uf_new.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_faixa_valor_ju_wide_by_uf.csv')

In [135]:
df_cubo['faixa_vlr_pago_ju_bbagil'].unique().tolist()

['Acima de 200 mil',
 'De 10 a 50 mil',
 'De 50 a 200 mil',
 'De 2 a 10 mil',
 'Até 2 mil']

In [120]:
# aggregate_faixa_valor_ju_wide_by_state.csv
df_uf_new = territorial.aggregate_faixa_valor_ju_wide_by_uf(df_cubo=df_cubo, by_filter='ESTADO')
# df_uf_new.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_faixa_valor_ju_wide_by_state.csv')

In [153]:
df_uf_new

,uf,total_contemplados_uf,valor_total_uf,qtd_ate_2_mil,perc_qtd_ate_2_mil,valor_ate_2_mil,perc_valor_ate_2_mil,qtd_de_2_a_10_mil,perc_qtd_de_2_a_10_mil,valor_de_2_a_10_mil,perc_valor_de_2_a_10_mil,qtd_de_10_a_50_mil,perc_qtd_de_10_a_50_mil,valor_de_10_a_50_mil,perc_valor_de_10_a_50_mil,qtd_de_50_a_200_mil,perc_qtd_de_50_a_200_mil,valor_de_50_a_200_mil,perc_valor_de_50_a_200_mil,qtd_acima_de_200_mil,perc_qtd_acima_de_200_mil,valor_acima_de_200_mil,perc_valor_acima_de_200_mil
0,AC,1079,"24,329,783.680",86,0.080,"119,410.020",0.005,404,0.374,"2,713,109.200",0.112,539,0.500,"12,812,365.550",0.527,40,0.037,"3,525,137.400",0.145,10,0.009,"5,159,761.510",0.212
1,AL,6378,"60,294,994.130",2543,0.399,"3,045,933.360",0.051,2566,0.402,"12,627,446.230",0.209,1121,0.176,"25,239,552.430",0.419,129,0.020,"11,384,127.010",0.189,19,0.003,"7,997,935.100",0.133
2,AM,2075,"74,122,348.910",253,0.122,"388,629.700",0.005,575,0.277,"3,147,992.640",0.042,929,0.448,"27,779,370.170",0.375,289,0.139,"28,336,046.250",0.382,29,0.014,"14,470,310.150",0.195
3,AP,649,"23,633,686.160",181,0.279,"222,658.850",0.009,216,0.333,"1,381,748.300",0.058,187,0.288,"4,462,053.260",0.189,40,0.062,"3,970,638.100",0.168,25,0.039,"13,596,587.650",0.575
4,BA,17500,"176,909,114.840",5776,0.330,"7,798,862.070",0.044,8920,0.510,"42,307,782.740",0.239,2294,0.131,"58,629,214.980",0.331,438,0.025,"42,126,855.730",0.238,72,0.004,"26,046,399.320",0.147
5,CE,8256,"136,622,936.070",2739,0.332,"3,723,850.040",0.027,3561,0.431,"18,032,030.320",0.132,1478,0.179,"37,819,829.870",0.277,408,0.049,"43,411,131.190",0.318,70,0.008,"33,636,094.650",0.246
6,DF,393,"41,170,700.290",98,0.249,"133,862.920",0.003,91,0.232,"373,251.580",0.009,39,0.099,"1,071,727.780",0.026,136,0.346,"13,086,856.380",0.318,29,0.074,"26,505,001.630",0.644
7,ES,1883,"55,730,593.340",35,0.019,"47,574.130",0.001,916,0.486,"5,622,769.830",0.101,696,0.370,"17,359,185.210",0.311,210,0.112,"19,527,596.510",0.350,26,0.014,"13,173,467.660",0.236
8,GO,4320,"105,504,440.400",954,0.221,"1,218,562.190",0.012,2000,0.463,"10,967,483.440",0.104,890,0.206,"24,427,850.350",0.232,398,0.092,"40,947,519.240",0.388,78,0.018,"27,943,025.180",0.265
9,MA,8366,"114,921,030.050",3406,0.407,"4,624,102.760",0.040,3381,0.404,"16,070,620.220",0.140,1201,0.144,"28,781,224.320",0.250,308,0.037,"30,079,637.510",0.262,70,0.008,"35,365,445.240",0.308


In [20]:
# executed_value_by_region
df_states_region       = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='ESTADO')
df_municipality_region = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_uf_region           = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='UF')

# df_states_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_state.csv')
# df_municipality_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_municipality.csv')
# df_uf_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_uf.csv')


In [146]:
df_uf_region

,regiao,valor_executado_rs,qtde_contemplados,min_valor,mediana_valor,max_valor,media_valor,populacao,perc_valor_executado,perc_qtde_contemplados,perc_populacao,perc_contemplados_populacao,qtd_tipo_documento_CNPJ,qtd_tipo_documento_CPF,valor_tipo_documento_CNPJ,valor_tipo_documento_CPF,min_valor_tipo_documento_CNPJ,min_valor_tipo_documento_CPF,mediana_valor_tipo_documento_CNPJ,mediana_valor_tipo_documento_CPF,max_valor_tipo_documento_CNPJ,max_valor_tipo_documento_CPF,media_valor_tipo_documento_CNPJ,media_valor_tipo_documento_CPF
0,Centro-Oeste,"237,420,986.840",9335,378.000,"5,804.235","2,000,000.000","25,784.208",17071595,0.083,0.056,0.080,0.001,1898,7437,"116,959,750.110","120,461,236.730",386.700,378.000,"15,570.000","5,000.000","2,000,000.000","735,000.000","62,983.172","16,387.054"
1,Nordeste,"852,268,894.220",79446,380.000,"3,000.000","7,792,386.600","11,226.177",57112096,0.299,0.476,0.269,0.001,9063,70383,"393,254,434.490","459,014,459.730",380.000,380.000,"14,800.000","2,500.000","7,792,386.600","440,000.000","46,129.552","6,811.011"
2,Norte,"303,553,050.960",14504,400.000,"6,025.000","22,109,764.920","21,631.373",18669345,0.107,0.087,0.088,0.001,1576,12928,"153,453,413.560","150,099,637.400",450.000,400.000,"29,934.280","5,000.000","22,109,764.920","280,000.000","100,956.193","11,995.496"
3,Sudeste,"1,066,203,024.520",45655,375.000,"6,904.320","8,000,000.000","23,836.419",88617693,0.375,0.274,0.417,0.001,11917,33738,"668,453,933.040","397,749,091.480",375.000,375.000,"15,950.000","5,000.000","8,000,000.000","600,000.000","57,990.278","11,979.312"
4,Sul,"386,549,637.450",17946,375.000,"8,802.500","2,400,000.000","22,040.691",31113021,0.136,0.108,0.146,0.001,7826,10120,"259,190,161.370","127,359,476.080",375.000,380.000,"12,727.790","6,877.010","2,400,000.000","461,000.000","34,567.906","12,685.207"


In [118]:
# aggregate_values_by
df_municipality_agg     = territorial.aggregate_execution_summary_by_scope(df_cubo=df_cubo, scope='MUNICIPIO')
df_state_agg            = territorial.aggregate_execution_summary_by_scope(df_cubo=df_cubo, scope='ESTADO')
df_capital_agg          = territorial.aggregate_capital_interior_summary(df_cubo=df_cubo)

# df_municipality_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_municipality.csv')
# df_capital_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_capital.csv')
# df_state_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_state.csv')

In [ ]:
# values_by_population_size
df_population_size = territorial.aggregate_execution_by_porte_with_estado(df_cubo=df_cubo)
denominador_urbano_rural = (
    df_population_size["valor_urbano_por_porte"]
    + df_population_size["valor_rural_por_porte"]
)

df_population_size["percentual_valor_urbano_por_porte"] = np.where(
    denominador_urbano_rural.ne(0),
    df_population_size["valor_urbano_por_porte"] / denominador_urbano_rural,
    np.nan
)

df_population_size["percentual_valor_rural_por_porte"] = np.where(
    denominador_urbano_rural.ne(0),
    df_population_size["valor_rural_por_porte"] / denominador_urbano_rural,
    np.nan
)
# df_population_size.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_population_size.csv')

In [79]:
# resumo_por_porte_populacional versão RESUMIDA - Estamos utilizando esse
df_resumo_por_porte_populacional = territorial.resumo_por_porte_populacional(df_aux=df_aux)
df_resumo_por_porte_populacional.to_csv(settings.DATA_PATH_SECTION1 / 'resumo_por_porte_populacional.csv')


In [24]:
# resumo_valor_por_porte_municipio.csv
df_population_size_mean = territorial.resumo_valor_por_porte_municipio(df_cubo=df_cubo)
# df_population_size_mean.to_csv(settings.DATA_PATH_SECTION1 / 'population_size_mean.csv')

In [4]:
# values_by_special_territory
df_special_territory_municipality = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="MUNICIPIO"
)

df_special_territory_state = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="ESTADO"
)

df_special_territory_uf = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="UF"
)

# df_special_territory_municipality.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_municipality.csv')
# df_special_territory_state.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_state.csv')
# df_special_territory_uf.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_uf.csv')


In [86]:
# special_territory_w_ibge_by_brazil
df_vis_territorio_brasil = territorial.generate_special_territories_brazil_view(df_cubo=df_cubo)
# df_vis_territorio_brasil.to_csv(settings.DATA_PATH_SECTION1 / 'special_territory_w_ibge_by_brazil.csv')

In [88]:
# aggregate_by_local_residencia
df_interior_rm_uf = territorial.aggregate_by_local_residencia(df_cubo=df_cubo, visao='uf')
# df_interior_rm_uf.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_by_local_residencia_uf.csv')

In [96]:
df_interior_rm_estado = territorial.aggregate_by_local_residencia(df_cubo=df_cubo, visao='estado')

In [111]:
df_interior_por_uf_estado = territorial.aggregate_estado_by_uf_local_residencia(df_cubo=df_cubo)

In [115]:
#aggregate_estado_by_uf_local_residencia
df_interior_por_uf_estado.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_estado_by_uf_local_residencia.csv')

In [ ]:
# df_interior_rm_estado
df_interior_rm_estado.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_by_local_residencia_estado.csv')


# Section 2

In [27]:
df_values_by_person_type_uf = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='UF')
df_values_by_person_type_state = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='ESTADO')
df_values_by_person_type_municipality = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='MUNICIPIO')

df_values_by_person_type_uf.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_uf.csv')
df_values_by_person_type_state.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_state.csv')
df_values_by_person_type_municipality.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_municipality.csv')

In [123]:
df_faixa_valor = territorial.aggregate_faixa_valor_ju_by(df_cubo=df_cubo)
# df_faixa_valor.to_csv(settings.DATA_PATH_SECTION2 / 'values_range_by_brazil_v2.csv')

In [131]:
df_aux['nome_macrorregiao'].unique().tolist()

['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']

In [ ]:
df_faixa_valor = territorial.aggregate_faixa_valor_ju_by(df_cubo=df_cubo, by_filter='ESTADO')
df_faixa_valor.to_csv(settings.DATA_PATH_SECTION2 / 'values_range_by_state_v2.csv')

In [26]:
df_aux = pd.read_parquet(settings.DATA_PATH / 'input_data'/ 'non-public' / 'df_aux_cubo.parquet')

In [27]:
df_box = territorial.make_boxplot_df_faixa_valor(df_aux=df_aux)

In [125]:
df_box

,visao,uf_bbagil,faixa_vlr_pago_ju_bbagil,metrica,valor_boxplot,unidade_observacao
0,ESTADO,AC,Até 2 mil,quantidade_contemplados,13.000,uf_faixa
1,ESTADO,AC,De 2 a 10 mil,quantidade_contemplados,25.000,uf_faixa
2,ESTADO,AC,De 10 a 50 mil,quantidade_contemplados,368.000,uf_faixa
3,ESTADO,AC,De 50 a 200 mil,quantidade_contemplados,36.000,uf_faixa
4,ESTADO,AC,Acima de 200 mil,quantidade_contemplados,10.000,uf_faixa
...,...,...,...,...,...,...
22157,ESTADO,TO,Acima de 200 mil,valor_transacao_total_bbagil,"367,000.000",contemplado
22158,ESTADO,TO,Acima de 200 mil,valor_transacao_total_bbagil,"237,000.000",contemplado
22159,ESTADO,TO,Acima de 200 mil,valor_transacao_total_bbagil,"237,000.000",contemplado
22160,ESTADO,TO,Acima de 200 mil,valor_transacao_total_bbagil,"237,000.000",contemplado


In [71]:
df_box_qtd_contemplados = df_box[df_box['metrica'] == 'quantidade_contemplados']
df_box_qtd_contemplados.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_box_plot_qtd_contemplados_state.csv')

In [159]:
import pandas as pd

# Filtra apenas ESTADO
df_estado = df_aux.copy()

# Garante que a coluna de valor está numérica
df_estado["valor_transacao_total_bbagil"] = pd.to_numeric(
    df_estado["valor_transacao_total_bbagil"],
    errors="coerce"
)

# Remove valores nulos
serie_valor = df_estado["valor_transacao_total_bbagil"].dropna()

# Limite superior para média aparada
p99 = serie_valor.quantile(0.99)

# Média aparada: remove os 1% maiores valores
media_aparada_1pct = serie_valor[serie_valor <= p99].mean()

# Tabela geral de percentis e quartis
df_percentis_estado = pd.DataFrame({
    "tipo_ente": ["GERAL"],
    "quantidade_contemplados": [serie_valor.count()],
    "valor_minimo": [serie_valor.min()],
    "p1": [serie_valor.quantile(0.01)],
    "q1": [serie_valor.quantile(0.25)],
    "q2_mediana": [serie_valor.quantile(0.50)],
    "q3": [serie_valor.quantile(0.75)],
    "p99": [p99],
    "valor_maximo": [serie_valor.max()],
    "media": [serie_valor.mean()],
    "media_aparada_1pct": [media_aparada_1pct],
    "desvio_padrao": [serie_valor.std()]
})

df_percentis_estado

,tipo_ente,quantidade_contemplados,valor_minimo,p1,q1,q2_mediana,q3,p99,valor_maximo,media,media_aparada_1pct,desvio_padrao
0,GERAL,166886,375.000,500.000,"2,000.000","4,950.740","12,500.000","200,000.000","22,109,764.920","17,053.531","12,855.080","106,478.885"


In [ ]:
df_box_qtd_contemplados = df_box[df_box['metrica'] == 'valor_t1ransacao_total_bbagil']
df_box_qtd_contemplados.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_box_plot_valor_transacao_total_bbagil_state.csv')

In [168]:
importlib.reload(territorial)

<module 'src.aggregations.territorial' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\territorial.py'>

In [39]:
# resumo_faixa_valor_por_porte.csv
df_resumo_faixas_porte = territorial.resumo_faixa_valor_por_porte(df_cubo=df_cubo)
df_resumo_faixas_porte.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_porte_populacional.csv')

In [40]:
df_resumo_faixas_porte

,porte_populacional,total_qtd_contemplados,total_valor_transacao,qtd_contemplados_acima_de_200_mil,perc_qtd_contemplados_acima_de_200_mil,valor_transacao_acima_de_200_mil,perc_valor_transacao_acima_de_200_mil,qtd_contemplados_de_10_a_50_mil,perc_qtd_contemplados_de_10_a_50_mil,valor_transacao_de_10_a_50_mil,perc_valor_transacao_de_10_a_50_mil,qtd_contemplados_de_50_a_200_mil,perc_qtd_contemplados_de_50_a_200_mil,valor_transacao_de_50_a_200_mil,perc_valor_transacao_de_50_a_200_mil,qtd_contemplados_de_2_a_10_mil,perc_qtd_contemplados_de_2_a_10_mil,valor_transacao_de_2_a_10_mil,perc_valor_transacao_de_2_a_10_mil,qtd_contemplados_ate_2_mil,perc_qtd_contemplados_ate_2_mil,valor_transacao_ate_2_mil,perc_valor_transacao_ate_2_mil
0,-99,22050,"1,450,514,326.10",1104,0.05,"592,272,160.59",0.41,11454,0.52,"317,106,059.64",0.22,4924,0.22,"509,229,282.04",0.35,4479,0.20,"31,762,705.32",0.02,89,0.00,"144,118.51",0.00
1,1_pequeno_i,57289,"266,445,332.42",7,0.00,"1,796,825.73",0.01,4511,0.08,"90,736,359.52",0.34,447,0.01,"33,425,238.51",0.13,23629,0.41,"106,275,294.55",0.40,28695,0.50,"34,211,614.11",0.13
2,2_pequeno_ii,38944,"234,896,275.23",22,0.00,"5,130,439.15",0.02,4424,0.11,"82,205,026.90",0.35,335,0.01,"28,685,142.34",0.12,19876,0.51,"100,181,929.29",0.43,14287,0.37,"18,693,737.55",0.08
3,3_medio,17348,"161,197,608.53",13,0.00,"4,263,457.18",0.03,3913,0.23,"77,580,099.05",0.48,254,0.01,"20,626,727.08",0.13,9523,0.55,"53,553,579.58",0.33,3645,0.21,"5,173,745.64",0.03
4,4_grande,31255,"732,942,051.71",201,0.01,"114,515,571.73",0.16,13359,0.43,"328,691,074.04",0.45,2388,0.08,"206,663,615.15",0.28,12986,0.42,"80,044,107.45",0.11,2321,0.07,"3,027,683.34",0.00


In [41]:
df_resumo_faixas_porte_cut = df_resumo_faixas_porte[df_resumo_faixas_porte['porte_populacional'] != '-99']

In [67]:
df_territorios_especiais = territorial.resumo_territorios_especiais_por_uf(df_cubo=df_cubo)
df_territorios_especiais.to_csv(settings.DATA_PATH_SECTION2 / 'territorios_especiais_por_uf.csv')

In [63]:
df_territorios_especiais = territorial.resumo_territorios_especiais_por_uf(df_cubo=df_cubo, visao='ESTADO')
# df_territorios_especiais.to_csv(settings.DATA_PATH_SECTION2 / 'territorios_especiais_por_municipio.csv')

In [144]:
df_agg_faixas_regiao = territorial.aggregate_faixa_valor_ju_wide_by_regiao(df_cubo=df_cubo)

In [167]:
df_regioes = territorial.resumo_por_regiao(df_aux=df_aux)
df_regioes.to_csv(settings.DATA_PATH_SECTION2 / 'resumo_por_regiao_uf.csv')

In [ ]:
# tabela_resumo_estado_municipio.csv
df_resumo_est_mun = territorial.tabela_resumo_estado_municipio(df_aux=df_aux)
df_resumo_est_mun.to_csv(settings.DATA_PATH_SECTION2 / 'tabela_resumo_estado_municipio.csv')

In [171]:
22050/166885

0.13212691374299668

# Section 3

In [16]:
importlib.reload(person)

<module 'src.aggregations.person' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\person.py'>

In [7]:
# aggregate_contemplados_pf_pj_proportion.csv
df_person = person.aggregate_contemplados_pf_pj_proportion(df_cubo=df_cubo)
df_person.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_pf_pj_proportion.csv')

In [11]:
df_cubo['tipo_documento'].value_counts(dropna=False)

tipo_documento
CPF     130235
CNPJ     25363
Name: count, dtype: int64

In [8]:
# aggregate_contemplados_by_sexo_proportion.csv
df_sexo = person.aggregate_contemplados_by_sexo_proportion(df_cubo=df_cubo)
df_sexo.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_by_sexo_proportion.csv')

In [50]:
df_sexo.head(2)

,quantidade_contemplados,perc_quantidade_contemplados,valor_contemplados,perc_valor_contemplados,quantidade_contemplados_feminino,perc_quantidade_contemplados_feminino,valor_contemplados_feminino,perc_valor_contemplados_feminino,quantidade_contemplados_masculino,perc_quantidade_contemplados_masculino,valor_contemplados_masculino,perc_valor_contemplados_masculino
0,134593,1,1254423238,1,62943,0.467654,578195818,0.460926,71650,0.532346,676227421,0.539074


In [13]:
# aggregate_valor_quantity_by_age_group_sexo_wide
df_age_group = person.aggregate_valor_quantity_by_age_group_sexo_wide(df_cubo=df_cubo)
df_age_group.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_valor_quantity_by_age_group_sexo_wide.csv')

In [ ]:
# aggregate_value_quantity_by_age_group_region_wide
df_age_region = person.aggregate_value_quantity_by_age_group_region_wide(df_cubo=df_cubo)
df_age_region.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_value_quantity_by_age_group_region_wide.csv')

# Section 4

In [53]:
importlib.reload(labor)

<module 'src.aggregations.labor' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\labor.py'>

In [49]:
# aggregate_vinculo_formal_labor.csv
df_not_in_mercado = labor.aggregate_vinculo_formal_labor(df_cubo=df_cubo)
df_not_in_mercado.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor.csv')

In [36]:
df_not_in_mercado

,numero_contemplados_sem_vinculo_trabalho_formal,numero_contemplados_com_vinculo_trabalho_formal,numero_contemplados_total,percentual_contemplados_sem_vinculo_trabalho_formal,percentual_contemplados_com_vinculo_trabalho_formal,valor_pago_sem_vinculo_trabalho_formal,valor_pago_com_vinculo_trabalho_formal,valor_pago_total,percentual_valor_pago_sem_vinculo_trabalho_formal,percentual_valor_pago_com_vinculo_trabalho_formal
0,74696,59910,134606,0.5550,0.4450,"649,385,460.0100","605,298,441.4100","1,254,683,901.4200",0.5180,0.4820


In [40]:
# aggregate_vinculo_formal_labor_by_uf.csv
df_not_in_mercado_by_uf = labor.aggregate_vinculo_formal_labor_by_uf(df_cubo=df_cubo)
df_not_in_mercado_by_uf.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_uf.csv')

In [37]:
# aggregate_vinculo_formal_labor_by_region.csv
df_not_in_mercado_by_region = labor.aggregate_vinculo_formal_labor_by_region(df_cubo=df_cubo)
df_not_in_mercado_by_region.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_region.csv')


In [44]:
# aggregate_vinculo_formal_labor_by_sexo.csv
df_not_in_mercado_by_sexo = labor.aggregate_vinculo_formal_labor_by_sexo(df_cubo=df_cubo)
df_not_in_mercado_by_sexo.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_sexo.csv')

In [50]:
# aggregate_vinculo_formal_labor_by_age_group.csv
df_not_in_mercado_by_age_group = labor.aggregate_vinculo_formal_labor_by_age_group(df_cubo=df_cubo)
df_not_in_mercado_by_age_group.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_age_group.csv')

In [15]:
# aggregate_vinculo_formal_labor_by_raca_cor.csv
df_not_in_mercado_by_raca_cor = labor.aggregate_vinculo_formal_labor_by_raca_cor(df_cubo=df_cubo)
# df_not_in_mercado_by_raca_cor.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_raca_cor.csv')


In [36]:
# resumo_raca_cor_com_vinculo_rais.csv
df_raca_cor_att = labor.resumo_raca_cor_com_vinculo_rais(df_cubo=df_cubo)
df_raca_cor_att.to_csv(settings.DATA_PATH_SECTION4 / 'resumo_raca_cor_com_vinculo_rais.csv')


In [ ]:
# resumo_escolaridade_com_vinculo_rais.csv
df_escolaridade = labor.resumo_escolaridade_com_vinculo_rais(df_cubo=df_cubo)
df_escolaridade.to_csv(settings.DATA_PATH_SECTION4 / 'resumo_escolaridade_com_vinculo_rais.csv')

In [45]:
df_escolaridade[['escolaridade_agregado_rais', 'valor_medio_transacao_com_vinculo']]

,escolaridade_agregado_rais,valor_medio_transacao_com_vinculo
0,Sem instrução e fundamental incompleto,"5,695.64"
1,Fundamental completo e médio incompleto,"6,635.20"
2,Médio completo e superior incompleto,"8,330.71"
3,Superior completo,"12,195.18"
4,Mestrado ou doutorado completo,"18,507.30"


In [10]:
df_cubo[df_cubo['raca_cor_desc_description'] == 'Parda – para a pessoa que se enquadrar como parda ou se declarar como mulata, cabocla, cafuza, mameluca ou mestiça de preto com pessoa de outra cor ou raça.']['quantidade'].sum()

np.int64(27071)

In [ ]:
# aggregate_raca_cor_vinculo_formal_labor_by_sexo
df_not_in_mercado_by_raca_cor_sexo = labor.aggregate_raca_cor_vinculo_formal_labor_by_sexo(df_cubo=df_cubo)
df_not_in_mercado_by_raca_cor_sexo.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_raca_cor_vinculo_formal_labor_by_sexo.csv')

In [54]:
# aggregate_vinculo_formal_labor_by_escolaridade.csv
df_not_in_mercado_escolaridade = labor.aggregate_vinculo_formal_labor_by_escolaridade(df_cubo=df_cubo)
# df_not_in_mercado_escolaridade.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_escolaridade.csv')

In [55]:
df_not_in_mercado_escolaridade

,escolaridade_agregado_rais,numero_contemplados_sem_vinculo_trabalho_formal,numero_contemplados_com_vinculo_trabalho_formal,numero_contemplados_total,percentual_contemplados_sem_vinculo_trabalho_formal,percentual_contemplados_com_vinculo_trabalho_formal,percentual_numero_contemplados_no_total_geral,percentual_numero_contemplados_sem_vinculo_no_total_geral,percentual_numero_contemplados_com_vinculo_no_total_geral,valor_pago_sem_vinculo_trabalho_formal,valor_pago_com_vinculo_trabalho_formal,valor_pago_total,percentual_valor_pago_sem_vinculo_trabalho_formal,percentual_valor_pago_com_vinculo_trabalho_formal,percentual_valor_pago_no_total_geral,percentual_valor_pago_sem_vinculo_no_total_geral,percentual_valor_pago_com_vinculo_no_total_geral
0,Sem informação,"74,426.00",0.00,"74,426.00",1.00,0.00,0.55,1.00,0.00,"646,871,967.68",0.00,"646,871,967.68",1.00,0.00,0.52,1.00,0.00
1,Médio completo e superior incompleto,0.00,"26,539.00","26,539.00",0.00,1.00,0.20,0.00,0.44,0.00,"221,132,646.28","221,132,646.28",0.00,1.00,0.18,0.00,0.36
2,Superior completo,0.00,"25,772.00","25,772.00",0.00,1.00,0.19,0.00,0.43,0.00,"313,612,032.62","313,612,032.62",0.00,1.00,0.25,0.00,0.52
3,Fundamental completo e médio incompleto,0.00,"3,716.00","3,716.00",0.00,1.00,0.03,0.00,0.06,0.00,"24,640,999.19","24,640,999.19",0.00,1.00,0.02,0.00,0.04
4,Sem instrução e fundamental incompleto,0.00,"2,224.00","2,224.00",0.00,1.00,0.02,0.00,0.04,0.00,"12,697,694.06","12,697,694.06",0.00,1.00,0.01,0.00,0.02
5,Mestrado ou doutorado completo,0.00,"1,929.00","1,929.00",0.00,1.00,0.01,0.00,0.03,0.00,"35,728,561.59","35,728,561.59",0.00,1.00,0.03,0.00,0.06


In [69]:
# aggregate_vinculo_trabalho_formal_by_escolaridade_clean.csv
df_not_in_mercado_escolaridade_clean = labor.aggregate_vinculo_trabalho_formal_by_escolaridade_sem_sem_informacao(df_cubo=df_cubo)
df_not_in_mercado_escolaridade_clean.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_trabalho_formal_by_escolaridade_clean.csv')

In [ ]:
# CBOS

np.float64(12415168.75)

In [56]:
# aggregate_cbo_rais.csv
df_cbo_rais = labor.aggregate_cbo_rais(df_cubo=df_cubo)
# df_cbo_rais.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_cbo_rais.csv')

In [57]:
df_cbo_rais

,cbo_descricao_rais,soma_quantidade,percentual_quantidade,soma_valor,percentual_valor
0,ASSISTENTE ADMINISTRATIVO,4936,0.08,"47,749,655.98",0.08
1,PROFESSOR DE NIVEL MEDIO NO ENSINO FUNDAMENTAL,2221,0.04,"26,266,309.39",0.04
2,AUXILIAR DE ESCRITORIO EM GERAL,2604,0.04,"22,148,387.76",0.04
3,DIRIGENTE DO SERVICO PUBLICO MUNICIPAL,2074,0.03,"21,190,394.39",0.03
4,PROFESSOR DE NIVEL SUPERIOR DO ENSINO FUNDAMEN...,1841,0.03,"18,610,977.54",0.03
...,...,...,...,...,...
1384,TECNICO EM MATERIAIS PRODUTOS CERAMICOS E VIDROS,1,0.00,712.72,0.00
1385,TECELAO DE MALHAS (MAQUINA CIRCULAR),1,0.00,500.00,0.00
1386,TECNICO EM MADEIRA,1,0.00,500.00,0.00
1387,DETETIVE PROFISSIONAL,1,0.00,407.56,0.00


# Section 5

In [24]:
importlib.reload(cadunico)

<module 'src.aggregations.cadunico' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\cadunico.py'>

In [47]:
df_cad_unico = pd.read_parquet(settings.DATA_PATH / 'input_data' / 'non-public' /'dim_cadunico__2026-05-19_18-17.parquet')

In [58]:
# aggregate_cadunico_summary.csv
df_cubo_cadunico = cadunico.aggregate_cadunico_summary(df_cubo=df_cubo)
# df_cubo_cadunico.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_summary.csv')

In [59]:
df_cubo_cadunico

,perc_contemplados_cadunico,qtd_contemplados_cadunico,qtd_documentos_unicos_cadunico,valor_recebido_cadunico,perc_valor_cadunico
0,0.43,58407,57338,"365,789,773.53",0.29


In [101]:
# aggregate_cadunico_profile_summary.csv
df_cubo_cadunico_sexo_idade = cadunico.aggregate_cadunico_profile_summary(df_cubo=df_cubo)
sexo = df_cubo_cadunico_sexo_idade[df_cubo_cadunico_sexo_idade['dimensao'] == 'Sexo']
faixa_etaria = df_cubo_cadunico_sexo_idade[df_cubo_cadunico_sexo_idade['dimensao'] == 'Faixa etária']
sexo.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_profile_summary_by_sexo.csv')
faixa_etaria.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_profile_summary_by_faixa_etaria.csv')


In [ ]:
# aggregate_cadunico_faixa_etaria_by_sexo.csv
df_cubo_cadunico_sexo_idade_juntos = cadunico.aggregate_cadunico_faixa_etaria_by_sexo(df_cubo=df_cubo)
df_cubo_cadunico_sexo_idade_juntos.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_faixa_etaria_by_sexo.csv')

In [ ]:
# aggregate_cadunico_by_situacao_renda.csv
df_cad_unico_situacao_renda = cadunico.aggregate_cadunico_by_situacao_renda(df_cubo=df_cubo)
df_cad_unico_situacao_renda.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_situacao_renda.csv')


In [ ]:
# aggregate_cadunico_by_fx_renda_per_capita.csv
df_cad_unico_situacao_faixa_renda_percapita = cadunico.aggregate_cadunico_by_fx_renda_per_capita(df_cubo=df_cubo)
df_cad_unico_situacao_faixa_renda_percapita.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_fx_renda_per_capita.csv')

In [ ]:
# aggregate_cadunico_by_situacao_domicilio.csv
df_cad_unico_domicilio_situacao = cadunico.aggregate_cadunico_by_situacao_domicilio(df_cubo=df_cubo)
df_cad_unico_domicilio_situacao.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_situacao_domicilio.csv')

In [187]:
# aggregate_cadunico_by_population_size.csv
df_cad_unico_porte_populacional = cadunico.aggregate_cadunico_by_population_size(df_cubo=df_cubo)
df_cad_unico_porte_populacional.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_population_size.csv')


In [60]:
# aggregate_cadunico_by_uf.csv
df_cad_unico_by_uf = cadunico.aggregate_cadunico_by_uf(df_cubo=df_cubo)
# df_cad_unico_by_uf.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_uf.csv')

In [63]:
df_cad_unico_by_uf

,uf,qtd_contemplados_cadunico,qtd_contemplados_total_uf,perc_qtd_cadunico_brasil,perc_qtd_total_brasil,valor_contemplados_cadunico,valor_contemplados_total_uf,perc_valor_cadunico_brasil,perc_valor_total_brasil
0,BA,8629,15713,0.15,0.12,"42,920,208.14","103,688,620.92",0.12,0.08
1,PE,6224,12062,0.11,0.09,"34,869,304.64","92,429,310.59",0.10,0.07
2,MG,5790,17290,0.10,0.13,"40,741,509.63","162,053,698.71",0.11,0.13
3,PB,5292,9278,0.09,0.07,"18,514,005.58","44,521,802.91",0.05,0.04
4,CE,3935,7418,0.07,0.06,"28,699,794.43","75,669,105.66",0.08,0.06
5,MA,3893,7464,0.07,0.06,"17,397,799.21","44,469,134.33",0.05,0.04
6,PA,3471,6850,0.06,0.05,"20,955,312.97","51,903,775.13",0.06,0.04
7,PI,3187,5294,0.05,0.04,"8,123,633.39","18,821,684.59",0.02,0.02
8,AL,2778,5686,0.05,0.04,"11,446,161.39","33,093,938.50",0.03,0.03
9,RN,2774,4787,0.05,0.04,"12,845,543.63","28,243,213.70",0.04,0.02


In [172]:
# aggregate_cadunico_by_value_group.csv
df_cad_unic_faixa_valor = cadunico.aggregate_cadunico_by_value_group(df_cubo=df_cubo)
df_cad_unic_faixa_valor.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_value_group.csv')


In [173]:
# aggregate_bolsa_familia_summary.csv
df_cad_unico_bpf = cadunico.aggregate_bolsa_familia_summary(df_cubo=df_cubo)
df_cad_unico_bpf.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_bolsa_familia_summary.csv')

In [180]:
# aggregate_bpc_summary.csv
df_cad_unico_bpc = cadunico.aggregate_bpc_summary(df_cubo=df_cubo)
df_cad_unico_bpc.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_bpc_summary.csv')

In [181]:
df_cpf_receita = pd.read_parquet(settings.DATA_PATH / 'input_data' / 'non-public' /'dim_cpf_receita__2026_05-19-13_09.parquet')

In [183]:
df_cpf_receita[df_cpf_receita['sexo_receita_cpf'] == 'Feminino']['cpf_receita_cpf'].nunique()

61026

In [184]:
df_cpf_receita[df_cpf_receita['sexo_receita_cpf'] == 'Masculino']['cpf_receita_cpf'].nunique()

68615

In [186]:
61026/(61026+68615)

0.47073071019199175